In [ ]:
# multi class classification of FGL sources using tree categories
import numpy as np
import pandas as pd
from matplotlib import pyplot
#from scipy.stats import binom
print(np.__version__)

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.calibration import calibration_curve

import time
import os
import sys
if '../lib/' not in sys.path:
    sys.path.append('../lib/')
import dio3
import multi_fgl_utils as util
import plot_util as putil

In [ ]:
if 1:
    import importlib
    importlib.reload(util);


In [ ]:
# select the catalog and the case
save_all_preds = 1
cat_name = '4FGL-DR4' # '4FGL-DR3'
if cat_name == '4FGL-DR4':
    # cases for GU & KDE papers
    baseline_case = '4FGL-DR4_GMM_RF'
    #case = '4FGL-DR4_GMM_RF' # 49 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu' # 51 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord' # 49 train / test splits
    case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp_w' # 48 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_w' # 49 train / test splits
    #case = '4FGL-DR4_GMM_RF_no_bcu_spp_w' # 48 train / test splits
    
    #baseline_case = '4FGL-DR4_2class_RF' # '4FGL-DR4_GMM_RF', '4FGL-DR4_2class_RF'
    #case = '4FGL-DR4_2class_pulsars_AGNs'
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 47 train/test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_no_bcu_spp' # 47 train/test splits
    #baseline_case = '4FGL-DR4_GMM_RF_no_coord' # 51 train/test splits
    #case = '4FGL-DR4_GMM_RF_no_coord_w' # 51 train/test splits
if cat_name == '4FGL-DR3':
    #baseline_case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp'
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp' # 57 splits
    #case = '4FGL-DR3_GMM_RF_no_coord_no_bcu_spp_nmin50' # 57 splits
    baseline_case = '4FGL-DR3_3class_RF'
    case = '4FGL-DR3_3class_RF'
    #case = '4FGL-DR3_3class_spec_features_RF'

baseline = (case == baseline_case)
root_folder = '../'
config_folder = root_folder + 'config/'

# generic inputs
cdict = util.get_config_dict(#cat_name=cat_name, 
                             baseline_case=baseline_case, 
                             #baseline_fn=baseline_fn,
                             #cases_fn=cases_fn,
                             case=case, 
                             folder=config_folder)


cdict['root_folder'] = root_folder
folder_in = cdict['root_folder'] + cdict.get('folder_in', 'data_in/')
folder_out = cdict['root_folder'] + cdict.get('folder_out', 'data_out/')
folder = folder_out
Eind = int(cdict.get('Eind', 1000))
class_name = cdict.get('class_name', 'CLASS1')

assoc_sources = cdict.get('assoc_sources', 'all_assoc')
sep_model = cdict.get('sep_model', 'GMM')
#sep_model = 'RF' # 'GMM', 'RF'
nmin = cdict.get('nmin')
if nmin is not None: 
    nmin = int(nmin)
max_depth = int(cdict.get('max_depth', 5))
feature_mode = cdict.get('features', 'all_features')

alg = cdict.get('class_alg', 'RF')
train_weighting = cdict.get('train_weighting')
test_weighting = cdict.get('test_weighting')
weighted = (train_weighting is not None) or (test_weighting is not None)
#oversample_factor = 1 # used for 'resampled' training and testing
max_sample = int(cdict.get('max_sample', 7))
test_frac = float(cdict.get('test_frac', 0.3))
nav = int(cdict.get('nav', 10))


tree_alg = cdict.get('sep_model', 'GMM')
#tree_alg = 'RF' # 'stat', 'GMM', 'RF', '4class', '2class', 'RF_aakash'


# input data
tree_cat_fn = util.get_tree_cat_fn(cdict)
tree_cat_dict = dio3.loaddict(tree_cat_fn)
if util.exist_internal(tree_cat_dict):
    util.remove_internal(tree_cat_dict)
cat_tree = list(tree_cat_dict.keys())
nclasses = len(tree_cat_dict)
max_tree_cat_dict = util.tree2max_tree(tree_cat_dict)
dec_tree_cat_dict = util.max_tree2dec_tree(max_tree_cat_dict)
nodes = [str(key) for key in dec_tree_cat_dict.keys()]


xdf_fn = util.get_xdf_fn(cdict)
xdf = pd.read_csv(xdf_fn, index_col=0)
ns = len(xdf)

all_features = [name for name in xdf.columns if name.find('CLASS') == -1]
if feature_mode == 'all_features':
    features = all_features
elif feature_mode == 'no_coord_features':
    features = [feat for feat in all_features if feat.find('GL') == -1]
elif feature_mode == 'spec_features':
    # should be equivalent to Toby's features
    features = ['LP_index1000MeV', 'LP_beta', 'log10(Energy_Flux100)', 'log10(Variability_Index)']

nsrc_dict = util.get_nsrc_dict(xdf, class_column=class_name, classes='associated')
assoc_classes = list(nsrc_dict.keys())

# algorithm description string
pars_dict = util.pars_dict
pars_dict['NN']['nfeatures'] = len(features)
pars_dict['NN']['classes'] = nodes
pars_dict['NN']['epochs'] = cdict.get('epochs', 500)
alg_descr = util.get_alg_descr(cdict, pars_dict=pars_dict)

weights_fn = util.get_weights_fn(cdict)
if os.path.isfile(weights_fn):
    weights_df = pd.read_csv(weights_fn, index_col=0)
    sample_ratio_0 = util.get_sample_ratio(weights_df['unas_pdf'], weights_df['assoc_pdf'], binary=False, 
                                           max_sample=max_sample)
else:
    sample_ratio_0 = 1.

if weighted:
    weight_str = '_weighted'
    sample_ratio = sample_ratio_0
else:
    sample_ratio = None
    weight_str = ''
    
folder_out = '../results/%s_probabilistic_catalogs/' % util.get_cat_descr(cdict)
if not os.path.isdir(folder_out):
    os.makedirs(folder_out)
fn_out = folder_out + util.get_cat_descr(cdict) + '_prob_cat.csv'

#class_def_fn = folder_out + '4FGL-DR3_%iclass%s_groups_def.csv' % (nclasses, alg_descr)

#algs = ['RF', 'SVC', 'NN']
algs = ['RF', 'NN']
#algs = ['RF',]
print(nclasses)
print(tree_cat_fn)
print(fn_out)

In [ ]:
all_features

In [ ]:
dec_tree_cat_dict

In [ ]:
ys = np.zeros(ns, dtype='<U%i' % (max_depth + 1))
clf_mask = np.zeros(ns, dtype=bool)
for key, cls in dec_tree_cat_dict.items():
    cls_mask = util.get_cl_mask(cls, xdf, cname='CLASS1')
    ys[cls_mask] = key
    clf_mask[cls_mask] = True
    
X = xdf[features][clf_mask]
X_unas = xdf[features][~clf_mask]
y = ys[clf_mask]


classif_columns = ['%s_%s' % (node, alg) for node in nodes for alg in algs]
print(classif_columns)



In [ ]:
add_sample_weights_only = False
if add_sample_weights_only:
    cat_df = pd.read_csv(fn_out, index_col=0)
    cat_df['sample_weight'] = sample_ratio_0
    cat_df['sample_weight'][~clf_mask] = 1.
    print('Save the prob catalog to file:')
    print(fn_out)
    print(cat_df.columns)
    cat_df.to_csv(fn_out)
    print(cat_df[['CLASS1', 'sample_weight']])

In [ ]:
#print(cat_df[['CLASS1', 'sample_weight']][:40])
#print(np.max(cat_df['sample_weight']))

In [ ]:
y_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
y2_pred = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0.)
nn_df = pd.DataFrame(index=xdf.index, columns=classif_columns, data=0)


In [ ]:
low_lim = 5
if save_all_preds:
    all_folder_out = fn_out.replace('.csv', '_all_pred/')
    if not os.path.isdir(all_folder_out):
        os.mkdir(all_folder_out)

for alg in algs:
    j = -1
    print(alg)
    alg_clms = [clm for clm in classif_columns if clm.find(alg) > -1]
    while np.min(nn_df[alg_clms].values) < low_lim:
        if alg.startswith('NN'):
            time0 = time.time()
        if save_all_preds:
            classif_columns_loc = ['%s_%s' % (node, alg) for node in nodes]
            y_pred_loc = pd.DataFrame(index=xdf.index, columns=classif_columns_loc, data=0.)
        j += 1
        print(j, np.min(nn_df[alg_clms].values))
        clf = util.get_classifier(alg, pars_dict[alg])
        if 1:
            clf_inds = np.arange(len(xdf))[clf_mask]
            rstate = np.random.RandomState(seed=j)
            test_inds = rstate.choice(clf_inds, size=int(len(clf_inds) * test_frac), replace=False, p=None)
            train_inds = [i for i in clf_inds if i not in test_inds]
            
            # add weights to training samples
            if weighted:
                sample_ratio_train = sample_ratio.iloc[train_inds]
            else:
                sample_ratio_train = None
            # add weights to testing samples
            #pred_dict_all[j]['test_weights'] = sample_ratio.iloc[test_inds]

            X_train = xdf[features].iloc[train_inds]
            X_test = xdf[features].iloc[test_inds]
            y_train = ys[train_inds]
            y_test = ys[test_inds]
            
            #clf.fit(X_train, y_train, sample_weight=sample_ratio_train)
        else:
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_frac, 
                                                                random_state=j)
        if alg == 'NN_SK':
            clf.fit(X_train, y_train)
        else:
            clf.fit(X_train, y_train, sample_weight=sample_ratio_train)
            
        pred_test = clf.predict_proba(X_test).T
        pred_unas = clf.predict_proba(X_unas).T

        class_inds = [list(clf.classes_).index(node) for node in nodes]
        for i, node in enumerate(nodes):
            clm = '%s_%s' % (node, alg)
            src_names = X_test.index
            y_pred[clm].loc[src_names] += pred_test[class_inds[i]]
            y2_pred[clm].loc[src_names] += pred_test[class_inds[i]]**2
            nn_df[clm].loc[src_names] += 1
            src_names = X_unas.index
            y_pred[clm].loc[src_names] += pred_unas[class_inds[i]]
            y2_pred[clm].loc[src_names] += pred_unas[class_inds[i]]**2
            nn_df[clm].loc[src_names] += 1
            if save_all_preds:
                src_names = X_test.index
                y_pred_loc[clm].loc[src_names] = pred_test[class_inds[i]]
                src_names = X_unas.index
                y_pred_loc[clm].loc[src_names] = pred_unas[class_inds[i]]
                fn = all_folder_out + '%s_%i.npy' % (alg, j)
                np.save(fn, np.array(y_pred_loc))


        if alg.startswith('NN'):
            print('Step time: %.2f s' %  (time.time() - time0))


In [ ]:
y_pred_norm = y_pred / nn_df
y_std = np.sqrt((y2_pred - y_pred_norm**2 * nn_df) / (nn_df - 1.))

In [ ]:
#sdf.index = range(1, len(sdf) + 1)

In [ ]:
# output the catalog
df_out = xdf.copy()
for clm in classif_columns:
    df_out[clm] = y_pred_norm[clm]
    df_out[clm + '_std'] = y_std[clm]
df_out['sample_weight'] = sample_ratio_0
df_out['sample_weight'][~clf_mask] = 1. # put 1 for unassociated sources

print('Save the prob catalog to file:')
print(fn_out)
df_out.to_csv(fn_out)

if save_all_preds:
    fn = all_folder_out + 'ntest.npy'
    print(fn)
    nns = np.array(nn_df[nn_df.columns[0]])
    print(nns[:10])
    np.save(fn, nns)


In [ ]:
df_out.columns

In [ ]:
sum(df_out['CLASS1'] == 'unk')

In [ ]:
sample_ratio_train

In [ ]:
dec_tree_cat_dict